# 03 — Optimisers Compared: SGD → Momentum → Adagrad → RMSProp → Adam

## What this notebook covers

We train the **exact same network** on the spiral dataset with all four optimisers,
then plot loss and accuracy curves side-by-side so you can see the real differences.

---

## Quick reference

| Optimiser | Key idea | Typical issue |
|-----------|----------|---------------|
| **SGD** | Move in gradient direction | Slow, sensitive to lr |
| **SGD + Momentum** | Accumulate velocity across steps | Can overshoot |
| **Adagrad** | Shrink lr for frequent params | lr → 0 eventually |
| **RMSProp** | Exponential moving avg of grad² | Fixes Adagrad's decay |
| **Adam** | Momentum + RMSProp + bias correction | 🏆 Best default choice |

---

## The math in one line each

**SGD:** $w \leftarrow w - \eta \nabla w$

**Momentum:** $v \leftarrow \beta v - \eta \nabla w \;,\quad w \leftarrow w + v$

**Adagrad:** $w \leftarrow w - \dfrac{\eta}{\sqrt{G + \varepsilon}} \nabla w \;,\quad G \leftarrow G + (\nabla w)^2$

**RMSProp:** same as Adagrad but $G \leftarrow \rho G + (1-\rho)(\nabla w)^2$

**Adam:** $m \leftarrow \beta_1 m + (1-\beta_1)\nabla w \;,\quad v \leftarrow \beta_2 v + (1-\beta_2)(\nabla w)^2 \;,\quad w \leftarrow w - \dfrac{\eta\,\hat{m}}{\sqrt{\hat{v}}+\varepsilon}$


## Setup

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import nnfs
from nnfs.datasets import spiral_data
nnfs.init()

from core import DenseLayer, ReLU, SoftmaxWithCrossEntropy
from core import SGD, Adagrad, RMSProp, Adam

X, y = spiral_data(samples=100, classes=3)
EPOCHS = 5001
LOG_EVERY = 500


## Training loop (reusable function)

In [ ]:
def train(optimizer, epochs=EPOCHS, log=True):
    """Train a fresh 2-layer network with the given optimiser."""
    np.random.seed(42)
    dense1      = DenseLayer(2, 64)
    activation1 = ReLU()
    dense2      = DenseLayer(64, 3)
    loss_fn     = SoftmaxWithCrossEntropy()

    history = {'loss': [], 'acc': [], 'lr': []}

    for epoch in range(epochs):
        # Forward
        dense1.forward(X)
        activation1.forward(dense1.output)
        dense2.forward(activation1.output)
        loss = loss_fn.forward(dense2.output, y)

        preds = np.argmax(loss_fn.output, axis=1)
        acc   = np.mean(preds == y)

        history['loss'].append(loss)
        history['acc'].append(acc)
        history['lr'].append(optimizer.current_learning_rate)

        if log and epoch % LOG_EVERY == 0:
            print(f'  epoch {epoch:>5d} | loss {loss:.4f} | acc {acc:.3f} | lr {optimizer.current_learning_rate:.6f}')

        # Backward
        loss_fn.backward(loss_fn.output, y)
        dense2.backward(loss_fn.dinputs)
        activation1.backward(dense2.dinputs)
        dense1.backward(activation1.dinputs)

        # Update
        optimizer.pre_update_params()
        optimizer.update_params(dense1)
        optimizer.update_params(dense2)
        optimizer.post_update_params()

    return history


## Run all optimisers

In [ ]:
configs = {
    'SGD':          SGD(learning_rate=1.0,   decay=1e-3),
    'SGD+Momentum': SGD(learning_rate=1.0,   decay=1e-3, momentum=0.9),
    'Adagrad':      Adagrad(learning_rate=1.0,  decay=1e-4),
    'RMSProp':      RMSProp(learning_rate=0.02, decay=1e-5, rho=0.999),
    'Adam':         Adam(learning_rate=0.05,  decay=5e-7),
}

results = {}
for name, opt in configs.items():
    print(f'\n── {name} ──')
    results[name] = train(opt)


## Side-by-side comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db', '#9b59b6']

for (name, hist), color in zip(results.items(), colors):
    axes[0].plot(hist['loss'], label=name, color=color)
    axes[1].plot(hist['acc'],  label=name, color=color)

axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[1].set_title('Training Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')

for ax in axes:
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Optimiser Comparison — Spiral Dataset', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


## What to observe

- **SGD** is the slowest — especially in early epochs where the gradient direction changes often.
- **Momentum** converges faster than vanilla SGD; it builds speed in consistent directions.
- **Adagrad** starts fast but can stall late training because its accumulated gradient never shrinks.
- **RMSProp** fixes that with the exponential moving average — stays adaptive throughout.
- **Adam** usually wins on both speed and final accuracy. It's the safe default for new projects.

> **Rule of thumb:** Start with Adam. If you need interpretability or are fine-tuning a large model, try SGD with momentum.
